In [ ]:
!pip install -q spacy
!python -m spacy download en_core_web_sm -q

import spacy
from spacy import displacy
from IPython.display import HTML, display

nlp = spacy.load("en_core_web_sm")

doc = nlp("The quick brown fox jumped over the lazy dogs.")

# jupyter=False returns the HTML, which we then display. (jupyter=True breaks on newer versions of IPython)
display(HTML(displacy.render(doc, style='dep', jupyter=False, options={'distance': 90})))

## Parsing and Part-of-Speech Tagging

Each token gets a part of speech (`pos_`) and a dependency relation (`dep_`) to its head word. Together these form the dependency tree drawn above.

In [ ]:
for token in doc:
    print(token.text, token.pos_, token.dep_)

In [ ]:
# A little more detail: the lemma, the fine-grained tag, and the head each word attaches to
for token in doc:
    print(f"{token.text:<8} {token.lemma_:<8} {token.pos_:<6} {token.tag_:<5} {token.dep_:<7} -> {token.head.text}")

# Not sure what a label means? Ask spaCy
print()
print("amod  =", spacy.explain("amod"))
print("pobj  =", spacy.explain("pobj"))
print("VBD   =", spacy.explain("VBD"))

In [ ]:
# The parse gives us structure we can use directly, e.g. noun phrases ("noun chunks")
# and who did what to whom (subject - verb - object)
doc2 = nlp("Robert gave the students a short lecture on parsing in Dublin.")

print("Noun chunks:", [chunk.text for chunk in doc2.noun_chunks])

for token in doc2:
    if token.dep_ == "ROOT":
        subj = [w.text for w in token.children if w.dep_ == "nsubj"]
        dobj = [w.text for w in token.children if w.dep_ == "dobj"]
        iobj = [w.text for w in token.children if w.dep_ == "dative"]
        print(f"{token.lemma_}(subject={subj}, object={dobj}, recipient={iobj})")

display(HTML(displacy.render(doc2, style='dep', jupyter=False, options={'distance': 90})))

## Named Entity Recognition

Note the first result below. Is *fox* really an organisation? Statistical models make mistakes, especially on short sentences with little context.

In [ ]:
for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)
print("there were this many entities: ", len(doc.ents))

In [ ]:
doc = nlp("Mickey Mouse walked along the River Liffey in Dublin, Ireland.")
for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)
print("there were this many entities: ", len(doc.ents))

## Measuring Correctness: the F1 score

In [ ]:
# Import statistics Library
import statistics

# Calculate F1 score / harmonic mean (assuming Precision and Recall already computed)
print(statistics.mean([0.5,0.7]))
print(statistics.harmonic_mean([0.5,0.7]))

# Now calculate the F1 score for (0.3,0.7).   (0.1,0.7), and (0.7,0.7)

Produce a visualization of the NER results
Important, spacy provides this ability baked in

1. Use a tool to download / scrape the content of a web page to get the text only, e.g., the wikipedia article for Dublin

2. With that data (or today if you don't know how to do that yet, copy and paste from a page) save the document to somewhere that colab can find it, and load the document for processing.

3. Apply an analysis to the data.

## Coreference Resolution

Coreference resolution links together the expressions in a text that refer to the same thing, e.g. *Mary* ... *she* ... *her*.

As mentioned in the lecture, tool support for this is still a little flaky. spaCy's own experimental coreference component (`spacy-experimental`) has not been updated for recent versions of spaCy and Python, so it will not install on Colab. Instead we use **fastcoref**, a small neural model that plugs into a spaCy pipeline.

First, to see why this is a hard problem, here is a very naive *white box* approach: link every pronoun to the most recent person mentioned before it.

In [ ]:
text = "Mary met John in Dublin. She gave him a book because he had helped her with her thesis."
doc = nlp(text)

pronouns = {"he", "him", "his", "she", "her", "hers"}
people = [ent for ent in doc.ents if ent.label_ == "PERSON"]

for token in doc:
    if token.lower_ in pronouns:
        earlier = [p for p in people if p.end <= token.i]
        guess = earlier[-1].text if earlier else "?"
        print(f"{token.text:<5} -> {guess}")

How many of those did the naive rule get right? What extra knowledge would a rule need to fix the mistakes (gender, number, grammatical role...)? This is why coreference is now handled by trained models.

In [ ]:
# fastcoref needs an older version of the transformers library.
# Ignore any pip warnings about dependency conflicts.
# If the import in the next cell fails: Runtime > Restart session, then run the next cell again.
!pip install -q fastcoref "transformers<5" "datasets<4"

In [ ]:
from fastcoref import spacy_component   # registers the "fastcoref" component with spaCy

nlp_coref = spacy.load("en_core_web_sm")
nlp_coref.add_pipe("fastcoref")          # downloads the model the first time it runs

doc = nlp_coref(text, component_cfg={"fastcoref": {"resolve_text": True}})

# Each cluster is a group of character spans that refer to the same entity
for cluster in doc._.coref_clusters:
    print([text[start:end] for start, end in cluster])

print()
print("Original:", text)
print("Resolved:", doc._.resolved_text)

In [ ]:
# Try some harder examples. Where does the model go wrong?
examples = [
    "The trophy didn't fit in the suitcase because it was too big.",
    "The trophy didn't fit in the suitcase because it was too small.",
    "TU Dublin announced a new research hub. The university said it will open next year.",
]
for ex in examples:
    d = nlp_coref(ex, component_cfg={"fastcoref": {"resolve_text": True}})
    print(d._.resolved_text)

## Dialogue Acts

In a simplified world there is a tight link between sentence type and dialogue act: a statement is an *Inform*, a question is an *Ask*, and an imperative is a *Request*. We can use the parse to build a baseline dialogue act tagger.

In [ ]:
def dialogue_act(sent):
    first = sent[0]
    root = sent.root
    if sent.text.strip().endswith("?") or first.tag_ in ("WDT", "WP", "WP$", "WRB"):
        return "Ask"
    # Imperative: a base-form verb as the root, with no subject ("Put the dinner on.")
    if root.tag_ == "VB" and not any(w.dep_ in ("nsubj", "nsubjpass") for w in root.children):
        return "Request"
    return "Inform"

dialogue = [
    "I'm starving, have you put the dinner on?",
    "No, I haven't had a chance, the dog barfed in his bed, and I just spent the last 20 minutes cleaning up that mess.",
    "When are you leaving?",
    "Put the kettle on.",
    # Indirect dialogue acts
    "It's gone very cold in here.",
    "I could do with a cup of tea.",
]

for turn in dialogue:
    for sent in nlp(turn).sents:
        print(f"{dialogue_act(sent):<8} {sent.text}")

The last two are tagged *Inform*, but most people would hear them as requests. Indirect dialogue acts need context and knowledge of conversational norms, not just grammar.

## Intent Recognition

In a conversational system we usually don't need a full dialogue act analysis. We just need to know what the user wants (the **intent**) and pick out the details we need to act on it (the **slots**). The domain limits what users are likely to ask, which makes the problem much easier.

### A white box approach: rules

spaCy's `Matcher` lets us write patterns over tokens, lemmas, parts of speech etc. Because we match on lemmas, *book*, *booked* and *booking* all match the same rule. Entities fill the slots, and the parse tells us which city is the origin and which is the destination.

In [ ]:
from spacy.matcher import Matcher

matcher = Matcher(nlp.vocab)
matcher.add("book_flight",   [[{"LEMMA": {"IN": ["book", "reserve", "need", "want"]}}, {"OP": "*"}, {"LEMMA": {"IN": ["flight", "ticket"]}}],
                              [{"LEMMA": "fly"}]])
matcher.add("check_weather", [[{"LEMMA": {"IN": ["weather", "forecast", "rain", "sunny"]}}]])
matcher.add("play_music",    [[{"LEMMA": "play"}, {"OP": "*"}, {"LEMMA": {"IN": ["song", "music", "album", "playlist"]}}]])

def rule_intent(text):
    doc = nlp(text)
    intents = {nlp.vocab.strings[match_id] for match_id, start, end in matcher(doc)}
    intent = intents.pop() if len(intents) == 1 else ("unknown" if not intents else "ambiguous")

    slots = {}
    for ent in doc.ents:
        if ent.label_ == "GPE":
            # Use the parse: is the city attached to "from" or "to"?
            role = {"from": "origin", "to": "destination"}.get(ent.root.head.lower_, "location")
            slots[role] = ent.text
        elif ent.label_ in ("DATE", "TIME"):
            slots["when"] = ent.text
    return intent, slots

for utterance in ["I want to book a flight from Dublin to Paris next Friday",
                  "Will it rain in Galway tomorrow?",
                  "Play some music by Hozier",
                  "Can you get me a ticket to Berlin?"]:
    print(utterance, "\n   ->", *rule_intent(utterance), "\n")

Notice the last example fails: *get me a ticket* isn't covered by our rules. Rules are transparent and need no training data, but you have to anticipate every way a user might say something.

### A black box approach: train a classifier

The alternative is the supervised process from the lecture: collect and label some examples, then train a model. spaCy has a built-in text classifier (`textcat`) we can train on a handful of examples in a few seconds.

In [ ]:
import random
from spacy.training import Example

train_data = [
    ("Book me a flight to London", "book_flight"),
    ("I need a plane ticket to Rome on Monday", "book_flight"),
    ("Can I fly from Cork to Madrid?", "book_flight"),
    ("Reserve two seats on the morning flight", "book_flight"),
    ("I want a return ticket to Boston", "book_flight"),
    ("Find me flights to Lisbon next week", "book_flight"),
    ("What's the weather like in Dublin?", "check_weather"),
    ("Is it going to rain tomorrow?", "check_weather"),
    ("Will it be sunny at the weekend?", "check_weather"),
    ("How cold is it outside?", "check_weather"),
    ("Give me the forecast for Galway", "check_weather"),
    ("Is it windy in Cork today?", "check_weather"),
    ("Play some music", "play_music"),
    ("Put on my workout playlist", "play_music"),
    ("I want to hear the new Taylor Swift album", "play_music"),
    ("Play Bohemian Rhapsody", "play_music"),
    ("Can you play something relaxing?", "play_music"),
    ("Play the latest song by Hozier", "play_music"),
]
labels = sorted({label for _, label in train_data})

nlp_intent = spacy.blank("en")
textcat = nlp_intent.add_pipe("textcat")
for label in labels:
    textcat.add_label(label)

examples = [Example.from_dict(nlp_intent.make_doc(text), {"cats": {l: float(l == label) for l in labels}})
            for text, label in train_data]

random.seed(0)
spacy.util.fix_random_seed(0)   # so everyone gets the same results
optimizer = nlp_intent.initialize(lambda: examples)
for epoch in range(20):
    random.shuffle(examples)
    losses = {}
    nlp_intent.update(examples, sgd=optimizer, losses=losses)
print("final training loss:", round(losses["textcat"], 4))

In [ ]:
tests = ["Can you get me a ticket to Berlin?",   # the one our rules missed
         "Is it raining in Galway?",
         "Play me something by Coldplay",
         "I'd love to listen to some jazz",
         "What is the capital of France?"]        # out of domain!

for text in tests:
    cats = nlp_intent(text).cats
    best = max(cats, key=cats.get)
    print(f"{text:<40} -> {best} ({cats[best]:.2f})")

The classifier handles the ticket request the rules missed, but look at the last two:

* *listen to some jazz* goes wrong because neither *listen* nor *jazz* appear in the training data. This model only knows the words it has seen. Representations like word embeddings (last week) help a model see that *listen* is related to *hear* and *music*.
* *What is the capital of France?* is outside the domain entirely, but the classifier always picks one of the intents it knows. Real systems add a confidence threshold or a fallback intent.

Real systems also need far more training data, and are evaluated on held-out examples using precision, recall and F1.

Notice also what the classifier does **not** give us: the slots. In practice, intent classification and entity/slot extraction are combined, which is exactly what we did with the rules above.

## Exercises

1. Take your Wikipedia text from the exercise above and run the coreference model on a paragraph of it. Count how many clusters it finds, and check a few by hand.
2. Add a new intent (e.g. `set_alarm`) to both the rule-based matcher and the trained classifier. Which was easier to extend? Write five new test utterances and compare the two approaches.